## 프로젝트에서의 데이터 활용 방안
대체 수종을 제안할 때는 '어느 지역에', '어떤 규모의 꿀벌들이','언제 먹이(밀원)가 부족한가'를 파악하는 것이 핵심입니다. 이 데이터셋은 그중 지역별 밀원 수요(꿀벌 규모)를 파악하는 기본 베이스 데이터로 활용됩니다.

지역별 양봉 밀집도 및 밀원 수요 분석 (수요처 타겟팅)

관할시도기관 및 관할시구기관별로 서양사육수와 토종사육수를 합산하여 어느 지역에 꿀벌이 가장 많이 살고 있는지(즉, 밀원 식물이 가장 절실한 지역이 어디인지) 파악합니다.

꿀벌 종류(서양종 vs 토종)에 따라 선호하거나 접근 가능한 밀원 식물의 특성이 다를 수 있으므로, 종별 사육 비율을 분석하여 대체 수종 선정의 디테일을 더할 수 있습니다.

- 외부 데이터와의 결합 (매핑 매개체)

이 데이터의 '시도/시구' 정보를 기준으로 기상청의 '지역별 개화 시기 데이터', '기온 데이터' 등을 결합(Merge)합니다.

이를 통해 "A 지역은 꿀벌 사육 수는 매우 많은데, 기온 상승으로 개화 시기가 앞당겨져 5월 초에 밀원 공백기가 발생하므로, 이 시기에 피는 대체 수종 X를 제안한다"라는 논리 구조를 완성할 수 있습니다.

In [5]:
import pandas as pd
import numpy as np
df = pd.read_csv('/content/drive/MyDrive/캡스톤_오데양/소스데이터셋/사용데이터(원본데이터)/농림축산식품부_양봉농가현황정보_20250819.csv')

print("=== 원본 데이터 정보 ===")
print(df.info())
print(df.head())

=== 원본 데이터 정보 ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 39 entries, 0 to 38
Data columns (total 17 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   농가일련번호  39 non-null     int64 
 1   고유번호    39 non-null     object
 2   관할시도기관  39 non-null     object
 3   관할시구기관  39 non-null     object
 4   접수번호    38 non-null     object
 5   접수일자    39 non-null     object
 6   처리기간    39 non-null     object
 7   농가구분    39 non-null     object
 8   법인일련번호  39 non-null     int64 
 9   서양사육수   39 non-null     int64 
 10  토종사육수   39 non-null     int64 
 11  신청구분코드  39 non-null     object
 12  신청구분    39 non-null     object
 13  신청발생일자  39 non-null     object
 14  신청일자    39 non-null     object
 15  신청사유    9 non-null      object
 16  비고      3 non-null      object
dtypes: int64(4), object(13)
memory usage: 5.3+ KB
None
   농가일련번호           고유번호   관할시도기관 관할시구기관           접수번호        접수일자 처리기간 농가구분  \
0   14546  성주군-서양종-01000     경상북도    성주군  성주군-서양종-01000  2

In [6]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
# 2. 프로젝트 목적에 필요한 핵심 컬럼만 추출
keep_columns = ['관할시도기관', '관할시구기관', '농가구분', '서양사육수', '토종사육수', '신청구분']
df_clean = df[keep_columns].copy()

In [8]:
# 3. 결측치 처리 및 데이터 타입 변환
# 사육수가 비어있는 경우 0으로 채우고 정수형(int)으로 변환
df_clean['서양사육수'] = df_clean['서양사육수'].fillna(0).astype(int)
df_clean['토종사육수'] = df_clean['토종사육수'].fillna(0).astype(int)

# 총 사육수 컬럼 생성 (지역별 전체 꿀벌 수요 파악용)
df_clean['총사육수'] = df_clean['서양사육수'] + df_clean['토종사육수']

# 지역명 텍스트 정제 (양 끝 공백 제거)
df_clean['관할시도기관'] = df_clean['관할시도기관'].str.strip()
df_clean['관할시구기관'] = df_clean['관할시구기관'].str.strip()

In [9]:
# 4. 프로젝트 핵심: 지역별(시도/시구) 꿀벌 사육 규모 집계 (Aggregation)
# 이 집계 결과를 바탕으로 향후 '개화시기 불일치 지역 데이터'와 결합하게 됩니다.
area_summary = df_clean.groupby(['관할시도기관', '관할시구기관']).agg(
    농가수=('총사육수', 'count'),
    총서양사육수=('서양사육수', 'sum'),
    총토종사육수=('토종사육수', 'sum'),
    전체사육수=('총사육수', 'sum')
).reset_index()

# 전체 사육수가 많은 순으로 정렬 (우선순위 지역 타겟팅용)
area_summary = area_summary.sort_values(by='전체사육수', ascending=False)


print("\n=== 전처리 및 지역별 집계 완료 ===")
print(area_summary.head(10))


=== 전처리 및 지역별 집계 완료 ===
     관할시도기관 관할시구기관  농가수  총서양사육수  총토종사육수  전체사육수
14     경상북도    성주군    2    1070       0   1070
2   강원특별자치도    횡성군    7     365     265    630
15     경상북도    안동시    1     450       0    450
8       경기도    화성시    4     431       0    431
10     경상남도    산청군    2     340       0    340
24     충청남도    홍성군    2     330       0    330
19  전북특별자치도    고창군    1     300       0    300
18     전라남도    보성군    1     210       0    210
9      경상남도    사천시    1     200       0    200
25     충청북도    청주시    1       0     200    200


In [17]:
# 2. 필요한 핵심 컬럼 추출 및 요청하신 영문명으로 변경
# 매핑 딕셔너리를 활용해 컬럼명을 직관적으로 변경합니다.
rename_dict = {
    '관할시도기관': 'sido',
    '관할시구기관': 'sigungu',
    '농가수': 'farm_count',
    '총서양사육수': 'western_bee',
    '총토종사육수': 'native_bee',
    '전체사육수': 'total_colonies'
}

# area_summary에 바로 rename을 적용하여 새로운 변수(또는 덮어쓰기)에 저장합니다.
area_summary_eng = area_summary.rename(columns=rename_dict)

print("\n=== 전처리, 집계 및 영문명 변경 완료 ===")
print(area_summary_eng.head(10))


=== 전처리, 집계 및 영문명 변경 완료 ===
       sido sigungu  farm_count  western_bee  native_bee  total_colonies
14     경상북도     성주군           2         1070           0            1070
2   강원특별자치도     횡성군           7          365         265             630
15     경상북도     안동시           1          450           0             450
8       경기도     화성시           4          431           0             431
10     경상남도     산청군           2          340           0             340
24     충청남도     홍성군           2          330           0             330
19  전북특별자치도     고창군           1          300           0             300
18     전라남도     보성군           1          210           0             210
9      경상남도     사천시           1          200           0             200
25     충청북도     청주시           1            0         200             200


In [18]:
# 영문 컬럼명이 적용된 데이터프레임을 CSV로 저장
# encoding='utf-8-sig'를 지정해야 엑셀(Excel)에서 열었을 때 한글(시도, 시군구명)이 깨지지 않습니다.
area_summary_eng.to_csv("양봉농가현황_전처리.csv", index=False, encoding='utf-8-sig')

print("저장이 완료되었습니다! 코랩 왼쪽 파일 메뉴에서 '양봉농가현황_전처리.csv'를 확인하세요.")

저장이 완료되었습니다! 코랩 왼쪽 파일 메뉴에서 '지역별_꿀벌사육현황_집계.csv'를 확인하세요.
